# Evaluasi Tengah Semester INF-1316 Organisasi dan Arsitektur Komputer
Dosen: Mulya Agung
## Simulator Data Hazards CPU dan Virtual Memory

**Petunjuk**
1. Isi `NIM` pada sel berikut, lalu jalankan semua sel dari atas (*Run All*).
2. **Jangan mengubah** sel generator soal dan sel konstanta. Parameter soal berbeda untuk setiap NIM.
3. Lengkapi setiap fungsi bertanda `# TODO` pada soal A-D dan **hapus baris `raise NotImplementedError`**. Jangan mengganti nama atau parameter fungsi.
4. Kode akan diuji otomatis dengan program grader menggunakan NIM Anda **dan** lima NIM lain, jadi fungsi harus bekerja untuk parameter apa pun.
5. Kumpulkan notebook ini ke eCampuz dengan nama `UTS_INF1316_<NIM>.ipynb`.

In [1]:
NIM = "51240057"

## Generator soal (jangan diubah)

In [2]:
# JANGAN DIUBAH: generator parameter soal per NIM
import random

PAGE_SIZE = 4096
SKELETONS = [  # register abstrak a..g dipetakan ke R1..R7 secara acak per NIM
    [("I1", "LOAD",  "a", [],         "m0"),
     ("I2", "ADD",   "b", ["a", "c"], None),
     ("I3", "SUB",   "a", ["d", "e"], None),
     ("I4", "STORE", None, ["b"],     "m1"),
     ("I5", "ADD",   "b", ["f", "g"], None),
     ("I6", "LOAD",  "c", [],         "m2"),
     ("I7", "STORE", None, ["c"],     "m3")],
    [("I1", "LOAD",  "a", [],         "m0"),
     ("I2", "SUB",   "b", ["c", "d"], None),
     ("I3", "ADD",   "c", ["a", "b"], None),
     ("I4", "LOAD",  "a", [],         "m1"),
     ("I5", "STORE", None, ["c"],     "m2"),
     ("I6", "ADD",   "e", ["a", "f"], None),
     ("I7", "STORE", None, ["e"],     "m3")],
    [("I1", "ADD",   "a", ["b", "c"], None),
     ("I2", "LOAD",  "b", [],         "m0"),
     ("I3", "STORE", None, ["a"],     "m1"),
     ("I4", "SUB",   "a", ["b", "d"], None),
     ("I5", "LOAD",  "c", [],         "m2"),
     ("I6", "ADD",   "e", ["a", "c"], None),
     ("I7", "STORE", None, ["e"],     "m3")],
]

def _outcome(program, pt):
    """Hitung jumlah page fault & protection fault (mode user) untuk validasi."""
    faults = prot = 0
    for _, op, _, _, addr in program:
        if addr is None: continue
        pte = pt[addr // PAGE_SIZE]
        w = op == "STORE"
        if pte["sup"] or (w and not pte["write"]) or (not w and not pte["read"]):
            prot += 1
        elif not pte["present"]:
            faults += 1
    return faults, prot

def generate(nim):
    rng = random.Random(int("".join(ch for ch in str(nim) if ch.isdigit())))
    while True:
        skeleton = rng.choice(SKELETONS)
        regs = rng.sample([f"R{i}" for i in range(1, 8)], 7)
        rmap = dict(zip("abcdefg", regs))
        vpns = rng.sample(range(4), 4)              # VPN untuk m0..m3
        addr = {f"m{k}": vpns[k] * PAGE_SIZE + rng.randrange(0, PAGE_SIZE, 4) for k in range(4)}
        program = [(n, op, rmap.get(d), [rmap[s] for s in src], addr.get(m))
                   for n, op, d, src, m in skeleton]
        frames = rng.sample(range(1, 16), 6)
        absent = set(rng.sample(range(4), 2))
        pt = {}
        for v in range(4):
            pt[v] = {"present": v not in absent, "ppn": None if v in absent else frames[v],
                     "read": 1, "write": rng.randint(0, 1), "sup": int(rng.random() < 0.25)}
        free = frames[4:6]
        f, p = _outcome(program, pt)
        if f >= 1 and p >= 1:                        # pastikan ada page fault & protection fault
            return program, pt, free

In [3]:
PAGE_FAULT_PENALTY = 100  # cycle

class PageFault(Exception): pass
class ProtectionFault(Exception): pass

PROGRAM, PAGE_TABLE, FREE_FRAMES = generate(NIM)

## Parameter soal Anda

In [4]:
print("Program:")
for name, op, dest, srcs, addr in PROGRAM:
    print(f"  {name}: {op:<5} tujuan={dest or '-':<4} sumber={', '.join(srcs) or '-':<8} alamat={hex(addr) if addr is not None else '-'}")
print("\nPage table (page size 4 KB, mode user):")
for vpn, e in PAGE_TABLE.items():
    print(f"  VPN {vpn}: present={int(e['present'])} ppn={e['ppn']} READ={e['read']} WRITE={e['write']} SUP={e['sup']}")
print("\nFREE_FRAMES =", FREE_FRAMES)

Program:
  I1: ADD   tujuan=R2   sumber=R6, R7   alamat=-
  I2: LOAD  tujuan=R6   sumber=-        alamat=0x5c0
  I3: STORE tujuan=-    sumber=R2       alamat=0x1a94
  I4: SUB   tujuan=R2   sumber=R6, R1   alamat=-
  I5: LOAD  tujuan=R7   sumber=-        alamat=0x3144
  I6: ADD   tujuan=R4   sumber=R2, R7   alamat=-
  I7: STORE tujuan=-    sumber=R4       alamat=0x2e84

Page table (page size 4 KB, mode user):
  VPN 0: present=1 ppn=9 READ=1 WRITE=0 SUP=1
  VPN 1: present=0 ppn=None READ=1 WRITE=1 SUP=0
  VPN 2: present=1 ppn=3 READ=1 WRITE=1 SUP=0
  VPN 3: present=0 ppn=None READ=1 WRITE=1 SUP=0

FREE_FRAMES = [2, 14]


## A. Deteksi data hazards (24 poin)
Kembalikan list tuple `(jenis, instr_awal, instr_akhir, register)` untuk **semua** pasangan (i, j) dengan i < j:
- **RAW**: register tujuan Ii dibaca Ij
- **WAR**: register sumber Ii ditulis Ij
- **WAW**: Ii dan Ij menulis register yang sama

In [5]:
def detect_hazards(program):
    hazards = []
    for i in range(len(program)):
        for j in range(i + 1, len(program)):
            name_i, op_i, dest_i, srcs_i, _ = program[i]
            name_j, op_j, dest_j, srcs_j, _ = program[j]

            # RAW: register tujuan Ii dibaca oleh Ij
            if dest_i is not None and dest_i in srcs_j:
                hazards.append(("RAW", name_i, name_j, dest_i))

            # WAR: register sumber Ii ditulis oleh Ij
            if dest_j is not None and dest_j in srcs_i:
                hazards.append(("WAR", name_i, name_j, dest_j))

            # WAW: Ii dan Ij menulis register yang sama
            if dest_i is not None and dest_j is not None and dest_i == dest_j:
                hazards.append(("WAW", name_i, name_j, dest_i))
    return hazards

for h in detect_hazards(PROGRAM):
    print(h)

('WAR', 'I1', 'I2', 'R6')
('RAW', 'I1', 'I3', 'R2')
('WAW', 'I1', 'I4', 'R2')
('WAR', 'I1', 'I5', 'R7')
('RAW', 'I1', 'I6', 'R2')
('RAW', 'I2', 'I4', 'R6')
('WAR', 'I3', 'I4', 'R2')
('RAW', 'I4', 'I6', 'R2')
('RAW', 'I5', 'I6', 'R7')
('RAW', 'I6', 'I7', 'R4')


## B. Register renaming (18 poin)
Ganti setiap register tujuan dengan register fisik baru `P0, P1, ...` sesuai urutan, dan perbarui register sumber pada instruksi berikutnya mengikuti pemetaan terbaru.

In [6]:
def rename_registers(program):
    mapping = {}
    renamed_program = []
    phys_reg_counter = 0

    for name, op, dest, srcs, addr in program:
        # Map source registers using the latest active renaming mapping
        new_srcs = [mapping.get(s, s) for s in srcs]

        # Map destination register if present
        new_dest = None
        if dest is not None:
            new_dest = f"P{phys_reg_counter}"
            mapping[dest] = new_dest
            phys_reg_counter += 1

        renamed_program.append((name, op, new_dest, new_srcs, addr))
    return renamed_program

try:
    renamed = rename_registers(PROGRAM)
    for ins in renamed:
        print(ins)
    print("Hazard tersisa:", detect_hazards(renamed))
except Exception as e:
    print("Error:", e)

('I1', 'ADD', 'P0', ['R6', 'R7'], None)
('I2', 'LOAD', 'P1', [], 1472)
('I3', 'STORE', None, ['P0'], 6804)
('I4', 'SUB', 'P2', ['P1', 'R1'], None)
('I5', 'LOAD', 'P3', [], 12612)
('I6', 'ADD', 'P4', ['P2', 'P3'], None)
('I7', 'STORE', None, ['P4'], 11908)
Hazard tersisa: [('RAW', 'I1', 'I3', 'P0'), ('RAW', 'I2', 'I4', 'P1'), ('RAW', 'I4', 'I6', 'P2'), ('RAW', 'I5', 'I6', 'P3'), ('RAW', 'I6', 'I7', 'P4')]


## C. Virtual memory (34 poin)
**`translate`** (24 poin): hitung VPN = `vaddr // PAGE_SIZE` dan offset. Periksa berurutan:
1. `SUP = 1` dan bukan supervisor → `ProtectionFault`
2. akses `"read"` dengan `READ = 0`, atau `"write"` dengan `WRITE = 0` → `ProtectionFault`
3. `present = False` → `PageFault(vpn)`

Kembalikan alamat fisik = `PPN * PAGE_SIZE + offset`.

**`handle_page_fault`** (10 poin): ambil frame kosong pertama dari `free_frames`, tulis sebagai PPN, set `present = True`.

In [7]:
def translate(vaddr, access, page_table, supervisor=False):
    vpn = vaddr // PAGE_SIZE
    offset = vaddr % PAGE_SIZE

    if vpn not in page_table:
        raise PageFault(vpn)

    pte = page_table[vpn]

    # 1. SUP = 1 dan bukan supervisor -> ProtectionFault
    if pte["sup"] == 1 and not supervisor:
        raise ProtectionFault("Supervisor access required")

    # 2. akses read/write sesuai hak akses
    if access == "read" and not pte["read"]:
        raise ProtectionFault("Read access denied")
    if access == "write" and not pte["write"]:
        raise ProtectionFault("Write access denied")

    # 3. present = False -> PageFault
    if not pte["present"]:
        raise PageFault(vpn)

    return pte["ppn"] * PAGE_SIZE + offset

def handle_page_fault(vpn, page_table, free_frames):
    if not free_frames:
        raise RuntimeError("No free frames available")
    frame = free_frames.pop(0)
    page_table[vpn]["present"] = True
    page_table[vpn]["ppn"] = frame

try:
    for name, op, _, _, addr in PROGRAM:
        if addr is None:
            continue
        access = "read" if op == "LOAD" else "write"
        try:
            print(name, hex(addr), "->", hex(translate(addr, access, PAGE_TABLE)))
        except PageFault as e:
            print(name, hex(addr), "-> PageFault pada VPN", e.args[0])
        except ProtectionFault as e:
            print(name, hex(addr), "-> ProtectionFault:", e)
except NameError:
    pass

I2 0x5c0 -> ProtectionFault: Supervisor access required
I3 0x1a94 -> PageFault pada VPN 1
I5 0x3144 -> PageFault pada VPN 3
I7 0x2e84 -> 0x3e84


## D. Eksekusi dan kinerja (24 poin)
**`raw_stalls`** (8 poin): stall setiap pasangan RAW = `max(0, 3 - jarak)`, dengan jarak = selisih indeks instruksi.

**`run`** (16 poin): jalankan program, kembalikan `(total_cycles, jumlah_page_fault, total_stall, log)`.
- 1 cycle per instruksi (termasuk yang dibatalkan), +100 cycle per page fault, ditambah total stall RAW.
- Setelah page fault: tangani, lalu **ulangi** instruksi.
- Protection fault: catat di log, instruksi dibatalkan, lanjut ke instruksi berikutnya.
- **Salin** `page_table` dan `free_frames` sebelum diubah; jangan mengubah milik pemanggil.

In [8]:
def raw_stalls(program):
    stalls = 0
    for i in range(len(program)):
        for j in range(i + 1, len(program)):
            name_i, op_i, dest_i, srcs_i, _ = program[i]
            name_j, op_j, dest_j, srcs_j, _ = program[j]
            if dest_i is not None and dest_i in srcs_j:
                jarak = j - i
                stalls += max(0, 3 - jarak)
    return stalls

import copy

def run(program, page_table, free_frames, supervisor=False):
    pt = copy.deepcopy(page_table)
    ff = copy.deepcopy(free_frames)
    log = []
    total_cycles = 0
    total_page_fault = 0
    total_stall = 0

    i = 0
    while i < len(program):
        name, op, dest, srcs, addr = program[i]
        total_cycles += 1 # 1 cycle per instruksi dasar

        # Hitung stall RAW untuk instruksi saat ini berdasarkan instruksi sebelumnya yang berhasil/tidak
        inst_stall = 0
        for prev_idx in range(max(0, i-2), i):
            p_name, p_op, p_dest, p_srcs, _ = program[prev_idx]
            if p_dest is not None and p_dest in srcs:
                jarak = i - prev_idx
                inst_stall = max(inst_stall, 3 - jarak)

        total_cycles += inst_stall
        total_stall += inst_stall

        if addr is not None:
            access = "read" if op == "LOAD" else "write"
            try:
                paddr = translate(addr, access, pt, supervisor)
                log.append(f"{name}: OK ({hex(paddr)})")
            except PageFault as e:
                vpn = e.args[0]
                total_cycles += 100
                total_page_fault += 1
                log.append(f"{name}: PageFault pada VPN {vpn}")
                handle_page_fault(vpn, pt, ff)
                # Ulangi instruksi saat ini dengan mengembalikan total_cycles dari run dasar agar tidak dihitung ganda
                total_cycles -= 1
                total_cycles -= inst_stall
                total_stall -= inst_stall
                continue
            except ProtectionFault as e:
                log.append(f"{name}: ProtectionFault - {str(e)}")
        else:
            log.append(f"{name}: OK")

        i += 1

    return total_cycles, total_page_fault, total_stall, log

try:
    cycles, faults, stalls, log = run(PROGRAM, PAGE_TABLE, FREE_FRAMES)
    for entry in log:
        print(entry)
    print(f"\npage fault = {faults}, stall RAW = {stalls}, total cycle = {cycles}")
except NameError:
    pass

I1: OK
I2: ProtectionFault - Supervisor access required
I3: PageFault pada VPN 1
I3: OK (0x2a94)
I4: OK
I5: PageFault pada VPN 3
I5: OK (0xe144)
I6: OK
I7: OK (0x3e84)

page fault = 2, stall RAW = 6, total cycle = 213
